# Automatic Transmission Model - Falsification of Specs using SpecForge SDK and Psy-Taliro

In this example notebook, we demonstrate how to perform falsification. The Falsification problem is as follows:
- Fixed System Model (i.e, a relation between input signals and output signals)
- Input: A Temporal Logic Specification on the input and output signals of the system
- Output: An Input Signal that causes the system to violate the specification

We use SpecForge to define, manage and monitor the specs; and we use [Psy-Taliro](https://psy-taliro.readthedocs.io/) to perform the falsification.

We will use a MATLAB/Simulink Model of an Automatic Transmission System as our model, which is commonly distributed with [MATLAB/Simulink](https://www.mathworks.com/help/simulink/slref/modeling-an-automatic-transmission-controller.html). This model is a commonly used benchmark for Falsification in the formal methods community.

## Setup

Import Necessary Libraries, including SpecForge SDK and Psy-Taliro

In [ ]:
import logging
import pathlib
import json
import os

import numpy as np
import plotly.graph_objects as go
import plotly.subplots as sp
import pandas as pd

# Toolbox for search-based test generation for cyber-physical systems
# See https://github.com/cpslab-asu/psy-taliro
from staliro import Sample, SignalInput, TestOptions, staliro
from staliro.models import Model, Result
from staliro.optimizers import DualAnnealing
from staliro.specifications import rtamt

from specforge_sdk import (
    SpecForgeClient,
    nested_encoding,
    flat_encoding,
    EXPORT_LILO,
    EXPORT_JSON,
    EXPORT_RTAMT,
    converters,
)

Running this example requires having MATLAB installed.

In [ ]:
import matlab
import matlab.engine

We should check if SpecForge is running

In [ ]:
# Get the port from environment variable or use default
port = os.environ.get("SPECFORGE_PORT", "8080")

# Initialize the client
specforgeClient = SpecForgeClient(base_url="http://localhost:" + port)

# Check connection
if specforgeClient.health_check():
    print(f"✓ Connected to SpecForge API v{specforgeClient.version()}")
else:
    print("✗ Cannot connect to SpecForge API")
    print("Make sure the SpecForge server is running on http://localhost:" + port)

# Model Definition

Now we set up our Model Class to simulate the Automatic Transmission model using MATLAB/Simulink.

In [ ]:
class AutotransModel(Model[list[float], None]):
    MODEL_NAME = "Autotrans_shift"

    def __init__(self) -> None:

        self.sampling_step = 0.2
        self.engine = matlab.engine.start_matlab()
        # Add the model path to MATLAB search path
        self.engine.addpath(
            str(specforgeClient.project_dir) + "/model/automatic_transmission/"
        )

        model_opts = self.engine.simget(AutotransModel.MODEL_NAME)
        self.model_opts = self.engine.simset(model_opts, "SaveFormat", "Array")

    def simulate(self, sample: Sample) -> Result[list[float], None]:
        assert sample.signals.tspan is not None

        tstart, tend = sample.signals.tspan
        duration = tend - tstart
        sim_t = matlab.double([0, tend])
        n_times = duration // self.sampling_step
        signal_times = np.linspace(tstart, tend, num=int(n_times))
        signal_values = np.array(
            [[signal.at_time(t) for t in signal_times] for signal in sample.signals]
        )

        model_input = matlab.double(
            np.row_stack((signal_times, signal_values)).T.tolist()
        )
        timestamps, _, data = self.engine.sim(
            self.MODEL_NAME, sim_t, self.model_opts, model_input, nargout=3
        )

        times: list[float] = np.array(timestamps).flatten().tolist()
        states: list[list[float]] = list(data)

        return Result(times=times, states=states, extra=None)


model = AutotransModel()

Use the Specforge SDK to translate the LILO specification to RTAMT format. The following formula establishes a condition on the speed based on the RPM of the engine.

In [ ]:
rtamt_formula = specforgeClient.export(
    system="automatic_transmission",
    definition="rpmSpeed4",
    export_type=EXPORT_RTAMT,
    return_string=True,  # Get the exported string
)
print("Exported RTAMT formula:", rtamt_formula)

## Setting up the Optimizer and Falsifier Options

Here we configure the details of the optimization process, including the choice of optimizer, number of iterations, and the mapping of the input to the system model.

In [ ]:
spec = rtamt.parse_discrete(rtamt_formula, {"rpm": 0, "speed": 1})

optimizer = DualAnnealing()

options = TestOptions(
    runs=1,
    iterations=100,
    tspan=(0, 30),
    signals={
        "throttle": SignalInput(control_points=[(0, 100)] * 7),
        "brake": SignalInput(control_points=[(0, 350)] * 3),
    },
)

## Running the Falsifier

This should take around 20-30 seconds on a typical laptop.

In [ ]:
logging.basicConfig(level=logging.DEBUG)
runs = staliro(model, spec, optimizer, options)
logging.getLogger().setLevel(logging.WARNING)


run = runs[0]
min_cost_eval = min(run.evaluations, key=lambda e: e.cost)
min_cost_trace = min_cost_eval.extra.trace

Let's pick the trace with the minimum cost, and convert it into a pandas DataFrame

In [ ]:
dataframe = pd.DataFrame(
    {
        "throttle": [
            min_cost_eval.sample.signals["throttle"].at_time(t)
            for t in min_cost_trace.times
        ],
        "brake": [
            min_cost_eval.sample.signals["brake"].at_time(t)
            for t in min_cost_trace.times
        ],
        "time": [t for t in min_cost_trace.times],
        "rpm": [s[0] for s in min_cost_trace.states],
        "speed": [s[1] for s in min_cost_trace.states],
        "gear": [s[2] for s in min_cost_trace.states],
    }
)
dataframe.head()

We can visualize the results by plotting the Signals

In [ ]:
figure = sp.make_subplots(rows=3, cols=1, shared_xaxes=True, x_title="Time (s)")

figure.add_trace(
    go.Scatter(x=dataframe["time"], y=dataframe["rpm"], name="RPM"), row=1, col=1
)

figure.add_trace(
    go.Scatter(x=dataframe["time"], y=dataframe["speed"], name="Speed"), row=2, col=1
)

figure.add_trace(
    go.Scatter(x=dataframe["time"], y=dataframe["gear"], name="Gear"), row=3, col=1
)

figure.update_yaxes(title_text="RPM", row=1, col=1)
figure.update_yaxes(title_text="Speed", row=2, col=1)
figure.update_yaxes(title_text="Gear", row=3, col=1)

figure.update_layout(title_text="Vehicle Trace (Outputs)")

figure.show()

In [ ]:
figure = sp.make_subplots(rows=3, cols=1, shared_xaxes=True, x_title="Time (s)")

figure.add_trace(
    go.Scatter(x=dataframe["time"], y=dataframe["throttle"], name="Throttle"),
    row=1,
    col=1,
)

figure.add_trace(
    go.Scatter(x=dataframe["time"], y=dataframe["brake"], name="Brake"), row=2, col=1
)

figure.add_trace(
    go.Scatter(x=dataframe["time"], y=dataframe["gear"], name="Gear"), row=3, col=1
)

figure.update_yaxes(title_text="Throttle", row=1, col=1)
figure.update_yaxes(title_text="Brake", row=2, col=1)
figure.update_yaxes(title_text="Gear", row=3, col=1)

figure.update_layout(title_text="Vehicle Trace (Inputs)")

figure.show()

We can also visualize the result using SpecForge's Debug Monitor 

In [ ]:
specforgeClient.monitor(
    system="automatic_transmission", definition="rpmSpeed4", data=dataframe
)